# Stage 6 (image, local GPU) — Fundus Image Classification, MobileNetV2 Transfer Learning
**Capstone: Multimodal Ocular Disease Prediction**

Runs entirely on your local machine (RTX 4060 8GB, i7-13th gen, 24GB RAM) in VS Code — **not Colab**. Uses PyTorch, because on Windows PyTorch has native pip-installable CUDA support, while TensorFlow dropped native Windows GPU support after 2.10 (it now needs WSL2) — PyTorch is the simpler, more reliable choice for a native Windows + VS Code setup.

Task: binary classification, **Myopia vs Normal**, using the full `datasets/myopia` and `datasets/normal` folders (already-resized 224x224 PNGs — the Stage 5 recommended fast-demo source).

**Before running this notebook:** run `gpu_check.py` in the same VS Code environment and confirm it reports a detected GPU. If it doesn't, fix that first — training will silently fall back to CPU otherwise (slow, and this notebook will print a loud warning if that happens, same honesty rule as before: no result is reported as a GPU transfer-learning result unless the GPU was actually used).

**Do not run this in the same environment as the earlier `06_cnn_proof_of_concept.ipynb` (the cloud-sandbox verification run) — that notebook used TensorFlow/Keras and a random-initialization fallback because that sandbox could not reach the pretrained-weights host. This notebook is the real one: PyTorch, real ImageNet weights (your machine has normal internet access), and the actual GPU.**

## 1. Confirm the GPU, then set the device

If this prints `cuda:0` you're good. If it prints `cpu`, stop and fix your PyTorch/CUDA install (see `gpu_check.py`) before continuing — training 3,000+ images on CPU will take much longer and is not what this stage is meant to demonstrate.

In [11]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("=" * 70)
    print("WARNING: no GPU detected -- training will run on CPU.")
    print("Run gpu_check.py and fix your PyTorch/CUDA install before trusting")
    print("any timing claims, and expect this notebook to take much longer.")
    print("=" * 70)


Using device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Locate the dataset

Points at the real `datasets/myopia` and `datasets/normal` folders in your project — no subsampling this time, the full class folders are used since local GPU training doesn't need the staging workaround the cloud-sandbox run needed.

In [12]:
import os

CANDIDATE_ROOTS = ["../..", ".", "../../.."]
DATA_ROOT = next((r for r in CANDIDATE_ROOTS if os.path.isdir(os.path.join(r, "datasets", "myopia"))), None)
if DATA_ROOT is None:
    raise FileNotFoundError("Could not find datasets/myopia. Set DATA_ROOT manually to your project folder.")

MYOPIA_DIR = os.path.join(DATA_ROOT, "datasets", "myopia")
NORMAL_DIR = os.path.join(DATA_ROOT, "datasets", "normal")

myopia_files = sorted(f for f in os.listdir(MYOPIA_DIR) if f.lower().endswith(".png"))
normal_files = sorted(f for f in os.listdir(NORMAL_DIR) if f.lower().endswith(".png"))

print("DATA_ROOT:", os.path.abspath(DATA_ROOT))
print("Myopia images:", len(myopia_files))
print("Normal images:", len(normal_files))
assert len(myopia_files) > 0 and len(normal_files) > 0, "No images found -- check DATA_ROOT."


DATA_ROOT: c:\Users\PRAJWAL\OneDrive\Desktop\Capstone Datasets
Myopia images: 232
Normal images: 2873


## 3. Train / validation / test split (stratified, image-level)

70/15/15, same rule as the cloud-sandbox verification run. Still image-level, not patient-level — `datasets/<class>` filenames carry no patient ID (Stage 5 finding), so a patient-grouped split isn't possible on this source.

In [13]:
from sklearn.model_selection import train_test_split

all_files = [(os.path.join(MYOPIA_DIR, f), 1) for f in myopia_files] + \
            [(os.path.join(NORMAL_DIR, f), 0) for f in normal_files]
paths = [p for p, _ in all_files]
labels = [l for _, l in all_files]

train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    paths, labels, test_size=0.30, random_state=42, stratify=labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.50, random_state=42, stratify=temp_labels
)

print("Train:", len(train_paths), "| positive rate:", round(sum(train_labels) / len(train_labels), 3))
print("Val:  ", len(val_paths), "| positive rate:", round(sum(val_labels) / len(val_labels), 3))
print("Test: ", len(test_paths), "| positive rate:", round(sum(test_labels) / len(test_labels), 3))


Train: 2173 | positive rate: 0.075
Val:   466 | positive rate: 0.075
Test:  466 | positive rate: 0.075


## 4. Dataset / DataLoader with augmentation

Normalization uses the standard ImageNet mean/std, matching how the pretrained MobileNetV2 was originally trained. Augmentation (flip, rotation, color jitter) is applied to the training set only.

In [14]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class FundusDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.float32)
        return img, label

train_ds = FundusDataset(train_paths, train_labels, train_transform)
val_ds = FundusDataset(val_paths, val_labels, eval_transform)
test_ds = FundusDataset(test_paths, test_labels, eval_transform)

BATCH_SIZE = 32
NUM_WORKERS = 0  # lower this to 0 if you hit DataLoader worker errors on Windows

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Batches per epoch -- train:", len(train_loader), "val:", len(val_loader), "test:", len(test_loader))


Batches per epoch -- train: 68 val: 15 test: 15


## 5. Build the model — MobileNetV2, real ImageNet weights, frozen backbone

Your machine has normal internet access, so `weights=MobileNet_V2_Weights.IMAGENET1K_V1` downloads the real pretrained weights here (unlike the cloud-sandbox run). The backbone is frozen (feature extraction) and a small trainable head replaces the original 1000-class classifier.

In [15]:
from torchvision import models
from torchvision.models import MobileNet_V2_Weights
import torch.nn as nn

weights = MobileNet_V2_Weights.IMAGENET1K_V1
base_model = models.mobilenet_v2(weights=weights)

for param in base_model.features.parameters():
    param.requires_grad = False  # freeze backbone

num_features = base_model.classifier[1].in_features
base_model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 64),
    nn.ReLU(),
    nn.Dropout(0.2),
    nn.Linear(64, 1),  # single logit -- BCEWithLogitsLoss handles the sigmoid
)

model = base_model.to(device)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} / {total_params:,} total")


Trainable parameters: 82,049 / 2,305,921 total


## 6. Train

In [16]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(train_labels)
class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=train_labels)
# pos_weight for BCEWithLogitsLoss = weight for the positive (Myopia) class relative to negative
pos_weight = torch.tensor([class_weights[1] / class_weights[0]], dtype=torch.float32).to(device)
print("Class weights (Normal, Myopia):", class_weights, "-> pos_weight:", pos_weight.item())

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

EPOCHS = 20
PATIENCE = 5
best_val_loss = float("inf")
epochs_no_improve = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_state = None

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels_ in train_loader:
        imgs, labels_ = imgs.to(device), labels_.to(device)
        optimizer.zero_grad()
        logits = model(imgs).squeeze(1)
        loss = criterion(logits, labels_)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = (torch.sigmoid(logits) >= 0.5).float()
        correct += (preds == labels_).sum().item()
        total += labels_.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    model.eval()
    val_running_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels_ in val_loader:
            imgs, labels_ = imgs.to(device), labels_.to(device)
            logits = model(imgs).squeeze(1)
            loss = criterion(logits, labels_)
            val_running_loss += loss.item() * imgs.size(0)
            preds = (torch.sigmoid(logits) >= 0.5).float()
            val_correct += (preds == labels_).sum().item()
            val_total += labels_.size(0)

    val_loss = val_running_loss / val_total
    val_acc = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch {epoch+1}/{EPOCHS} - train_loss: {train_loss:.4f} train_acc: {train_acc:.4f} "
          f"val_loss: {val_loss:.4f} val_acc: {val_acc:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1} (no val_loss improvement for {PATIENCE} epochs).")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    print("Restored best-val-loss weights.")


Class weights (Normal, Myopia): [0.54027847 6.70679012] -> pos_weight: 12.413579940795898
Epoch 1/20 - train_loss: 0.5420 train_acc: 0.8744 val_loss: 0.2850 val_acc: 0.9356
Epoch 2/20 - train_loss: 0.1967 train_acc: 0.9618 val_loss: 0.2365 val_acc: 0.9571
Epoch 3/20 - train_loss: 0.1873 train_acc: 0.9590 val_loss: 0.2342 val_acc: 0.9528
Epoch 4/20 - train_loss: 0.1470 train_acc: 0.9664 val_loss: 0.2529 val_acc: 0.9528
Epoch 5/20 - train_loss: 0.1854 train_acc: 0.9609 val_loss: 0.2164 val_acc: 0.9528
Epoch 6/20 - train_loss: 0.1588 train_acc: 0.9604 val_loss: 0.2361 val_acc: 0.9635
Epoch 7/20 - train_loss: 0.1707 train_acc: 0.9664 val_loss: 0.2349 val_acc: 0.9464
Epoch 8/20 - train_loss: 0.1384 train_acc: 0.9715 val_loss: 0.2152 val_acc: 0.9528
Epoch 9/20 - train_loss: 0.1140 train_acc: 0.9696 val_loss: 0.2595 val_acc: 0.9592
Epoch 10/20 - train_loss: 0.1418 train_acc: 0.9678 val_loss: 0.2667 val_acc: 0.9742
Epoch 11/20 - train_loss: 0.1868 train_acc: 0.9618 val_loss: 0.2766 val_acc: 0.

## 7. Training / validation curves

In [17]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history["train_loss"], label="train loss")
axes[0].plot(history["val_loss"], label="val loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train accuracy")
axes[1].plot(history["val_acc"], label="val accuracy")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.savefig("stage6_local_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved stage6_local_training_curves.png")


Saved stage6_local_training_curves.png


C:\Users\PRAJWAL\AppData\Local\Temp\ipykernel_4356\2018736703.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Evaluate on the held-out test set

In [18]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels_ in test_loader:
        imgs = imgs.to(device)
        logits = model(imgs).squeeze(1)
        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_preds.extend(preds.tolist())
        all_probs.extend(probs.tolist())
        all_labels.extend(labels_.numpy().astype(int).tolist())

acc = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, pos_label=1, zero_division=0)
rec = recall_score(all_labels, all_preds, pos_label=1, zero_division=0)
f1 = f1_score(all_labels, all_preds, pos_label=1, zero_division=0)
f1_macro = f1_score(all_labels, all_preds, average="macro", zero_division=0)

print(f"Device used for this run: {device}")
print(f"Test accuracy:  {acc:.4f}")
print(f"Precision (Myopia): {prec:.4f}")
print(f"Recall (Myopia):    {rec:.4f}")
print(f"F1 (Myopia):         {f1:.4f}")
print(f"F1 (macro):          {f1_macro:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=["Normal", "Myopia"], zero_division=0))

cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])
print("Confusion matrix (rows=actual, cols=predicted):")
print(cm)


Device used for this run: cuda
Test accuracy:  0.9657
Precision (Myopia): 0.6939
Recall (Myopia):    0.9714
F1 (Myopia):         0.8095
F1 (macro):          0.8953

              precision    recall  f1-score   support

      Normal       1.00      0.97      0.98       431
      Myopia       0.69      0.97      0.81        35

    accuracy                           0.97       466
   macro avg       0.85      0.97      0.90       466
weighted avg       0.97      0.97      0.97       466

Confusion matrix (rows=actual, cols=predicted):
[[416  15]
 [  1  34]]


In [19]:
import seaborn as sns

fig, ax = plt.subplots(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
            xticklabels=["Normal", "Myopia"], yticklabels=["Normal", "Myopia"])
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Test set confusion matrix (device={device})")
plt.tight_layout()
plt.savefig("stage6_local_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved stage6_local_confusion_matrix.png")


Saved stage6_local_confusion_matrix.png


C:\Users\PRAJWAL\AppData\Local\Temp\ipykernel_4356\699133543.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Save the model checkpoint

In [20]:
MODELS_DIR = os.path.join(DATA_ROOT, "Code", "models")
os.makedirs(MODELS_DIR, exist_ok=True)
model_path = os.path.join(MODELS_DIR, "stage6_myopia_vs_normal_mobilenetv2_local_gpu.pt")
torch.save(model.state_dict(), model_path)
print("Saved model state_dict to:", model_path)


Saved model state_dict to: ../..\Code\models\stage6_myopia_vs_normal_mobilenetv2_local_gpu.pt


## 10. Summary

| Item | Value |
|---|---|
| Task | Binary: Myopia vs Normal |
| Data source | Full datasets/myopia + datasets/normal folders (no subsampling) |
| Split | 70% train / 15% val / 15% test, stratified, image-level |
| Backbone | MobileNetV2, frozen, real ImageNet weights |
| Framework / device | PyTorch — printed `device` above records whether cuda or cpu actually ran |
| Augmentation | Random flip, rotation, color jitter (train only) |
| Imbalance handling | `pos_weight` in BCEWithLogitsLoss, computed from the training split |
| Training | Up to 20 epochs, early stopping on val loss (patience 5) |
| Evaluation | Accuracy, Precision/Recall/F1 (Myopia + macro), confusion matrix, training/validation curves — Sections 7-8 |
| Model checkpoint | Code/models/stage6_myopia_vs_normal_mobilenetv2_local_gpu.pt |

**Report back:** the printed `device` value, the final epoch count, and the Section 8 metrics/plots, so the master documentation can be updated with your real local-GPU results.